# 🎙️ Google Colab - CosyVoice 3.0 & XTTS v2 통합 GPU 서버

<a href="https://colab.research.google.com/github/ssss2513-cyber/ai-audio-studio/blob/main/CosyVoice_XTTS_Colab_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
### 💡 이 통합 서버의 특징
- **내 컴퓨터(4GB VRAM) 한계 완전 탈출**: 구글이 무료로 제공하는 **16GB VRAM (Tesla T4 GPU)** 환경에서 쾌적하게 구동됩니다.
- **완벽한 한국어 음성 복제 지원**:
  - **🔥 CosyVoice (알리바바)**: 감정 표현, 최고 음질 제로샷 한국어 음성 복제
  - **🦎 XTTS v2 (Coqui)**: 6초 샘플로 빠른 다국어/한국어 음성 복제
- **편리한 웹 UI**: 접속 가능한 공개 링크(Gradio / Cloudflare)가 즉시 발급되어 웹브라우저에서 바로 사용 가능합니다.

### ⚡ 사용 방법
1. 상단 메뉴 **런타임 ➔ 런타임 유형 변경**에서 **T4 GPU**가 선택되어 있는지 확인합니다.
2. **[1단계]** 환경 준비 셀을 실행합니다.
3. **[2단계 CosyVoice]** 또는 **[3단계 XTTS v2]** 중 사용하고 싶은 엔진의 셀을 실행하면 **접속 링크**가 출력됩니다!

In [ ]:
# [1단계] GPU 상태 확인 및 필수 시스템 패키지 설치
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg sox libsox-dev git-lfs curl wget
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("\n✅ [1단계] GPU 16GB 환경 확인 및 Cloudflared, FFmpeg, SoX 설치 완료!")

In [ ]:
# [2단계: 선택 A] 🔥 CosyVoice 3.0 / 2.0 고음질 한국어 복제 서버 구동
import os
import time
import subprocess

if not os.path.exists("/content/CosyVoice"):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice

%cd /content/CosyVoice

# 최신 Colab Python(3.11) 호환을 위한 고정 버전 완화 패치
!sed -i 's/onnxruntime-gpu==1.18.0/onnxruntime-gpu/g' requirements.txt
!sed -i 's/torch==2.3.1/torch/g' requirements.txt
!sed -i 's/torchaudio==2.3.1/torchaudio/g' requirements.txt

# 의존성 설치
!pip install -q onnxruntime-gpu
!pip install -q -r requirements.txt
!pip install -q gradio modelscope huggingface_hub soundfile torchaudio

# 모델 다운로드 (CosyVoice2-0.5B: 한국어 포함 9개 국어 지원)
from modelscope import snapshot_download
print("⏳ CosyVoice 사전학습 모델 다운로드 중... 잠시만 기다려주세요...")
snapshot_download("iic/CosyVoice2-0.5B", local_dir="pretrained_models/CosyVoice2-0.5B")
print("✅ 모델 다운로드 완료!")

# 이전 프로세스 청소
os.system("pkill -9 -f webui.py || true")
os.system("pkill -9 -f cloudflared || true")

# Cloudflare 터널 실행
tunnel_proc = subprocess.Popen(
    "cloudflared tunnel --url http://127.0.0.1:50000 --logfile /content/cosy_tunnel.log > /dev/null 2>&1",
    shell=True
)

print("⏳ 공개 접속 링크 생성 중...")
time.sleep(5)
public_url = None
if os.path.exists("/content/cosy_tunnel.log"):
    with open("/content/cosy_tunnel.log", "r") as f:
        import re
        for line in f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                public_url = m.group(0)
                break

print("\n" + "="*65)
if public_url:
    print(f"🎉 CosyVoice Cloudflare 접속 주소: {public_url}")
print("아래에 Gradio 공식 공유 링크도 함께 출력됩니다!")
print("="*65 + "\n")

# WebUI 구동 (share=True로 Gradio 링크 동시 발급)
!python3 webui.py --port 50000 --model_dir pretrained_models/CosyVoice2-0.5B


In [ ]:
# [3단계: 선택 B] 🦎 XTTS v2 (Coqui) 제로샷 한국어 복제 웹 서버 구동
import os
import time
import subprocess

%cd /content
# XTTS v2 라이브러리 및 WebUI 설치
!pip install -q coqui-tts gradio soundfile

# XTTS v2 제로샷 한국어 전용 WebUI 스크립트 작성
xtts_webui_code = '''import gradio as gr
import torch
import soundfile as sf
import os
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"XTTS v2 로딩 중... (장치: {device})")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("XTTS v2 로딩 완료!")

def clone_voice(ref_audio, text, language):
    if not ref_audio:
        return None, "⚠️ 참조 오디오 파일을 업로드해주세요."
    out_path = "output_xtts.wav"
    tts.tts_to_file(
        text=text,
        speaker_wav=ref_audio,
        language=language,
        file_path=out_path
    )
    return out_path, "✅ 생성 완료!"

demo = gr.Interface(
    fn=clone_voice,
    inputs=[
        gr.Audio(type="filepath", label="참조 음성 파일 (5~10초 한국어/영어 WAV, MP3)"),
        gr.Textbox(label="생성할 대사 내용", value="안녕하세요! XTTS v2 한국어 음성 복제 테스트입니다.", lines=3),
        gr.Dropdown(choices=["ko", "en", "ja", "zh", "es", "fr", "de"], value="ko", label="언어 선택")
    ],
    outputs=[
        gr.Audio(label="복제된 음성 결과"),
        gr.Textbox(label="진행 상태")
    ],
    title="🦎 XTTS v2 제로샷 목소리 복제 스튜디오 (Google Colab GPU)",
    description="구글 코랩 T4 GPU(16GB) 환경에서 구동되는 고속 한국어 음성 복제 서버입니다."
)

demo.launch(server_name="0.0.0.0", server_port=8080, share=True)
'''

with open("/content/xtts_app.py", "w", encoding="utf-8") as f:
    f.write(xtts_webui_code.strip())

!python3 /content/xtts_app.py
